# 5장 1강: A/B 테스트 설계 원리와 랜덤화 — 실습문제

## 실습 목표

- A/B 테스트의 대조군, 실험군, 랜덤화 단위를 데이터에서 식별합니다.
- 핵심 지표, 보조 지표, 가드레일 지표를 실험 목적에 맞게 사전에 정의합니다.
- 그룹 배정 수와 비율을 확인하고 계획한 50:50 배정과 일치하는지 점검합니다.
- 가설 → 설계 → 실행 → 분석의 순서로 온라인 실험계획을 작성합니다.

## 실습 환경 / 데이터

- Python, NumPy, pandas, SciPy
- `cookie_cats.csv`
- `userid`: 사용자 식별자
- `version`: 게임 게이트 위치(`gate_30`, `gate_40`)
- `sum_gamerounds`: 실험 기간 동안 플레이한 게임 라운드 수
- `retention_1`, `retention_7`: 설치 후 1일·7일 재방문 여부

> 이번 강의는 **실험 설계와 랜덤화 점검**이 중심입니다. 그룹 간 효과의 통계적 검정과 최종 배포 결정은 이후 강의에서 다룹니다.

## 실습 준비

아래 셀을 실행하여 데이터를 불러오고 크기, 결측치, 컬럼을 확인하세요.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

data_candidates = [
    Path("cookie_cats.csv"),
    Path("upload/cookie_cats.csv")]

data_path = next((path for path in data_candidates if path.exists()), None)

if data_path is None:
    raise FileNotFoundError("cookie_cats.csv 파일을 노트북과 같은 폴더에 넣어주세요.")

df = pd.read_csv(data_path)
alpha = 0.05

print(f"데이터 크기: {df.shape[0]}행, {df.shape[1]}열")
print("전체 결측치 수:", int(df.isna().sum().sum()))
print("컬럼:", df.columns.tolist())
df.head()


데이터 크기: 90189행, 5열
전체 결측치 수: 0
컬럼: ['userid', 'version', 'sum_gamerounds', 'retention_1', 'retention_7']


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


---

## 필수 1. Cookie Cats 실험 구조와 지표 정의

게임의 첫 번째 강제 대기 게이트를 30단계에서 40단계로 옮기면 사용자 유지율이 달라지는지 확인하려고 합니다.

### 수행 요구사항

1. `userid`의 중복 여부를 확인하여 사용자 한 명이 한 행으로 기록되었는지 점검하세요.
2. `version`의 고유값과 그룹별 사용자 수를 확인하세요.
3. 버전별 사용자 수, 평균 게임 라운드 수, 1일 유지율, 7일 유지율을 하나의 요약표로 만드세요.
4. 아래 질문에 문장으로 답하세요.

### 질문

1. 이 실험의 랜덤화 단위는 무엇인가요?
 - userid로 구분되는 사용자, 한 사용자는 하나의 버전에만 배정되어 일관된 게임 경험을 제공하여 비교

2. `gate_30`과 `gate_40` 중 대조군과 실험군은 각각 무엇으로 설정할 수 있나요?
 - 기존 게이트 위치인 gate_30을 대조군, 게이트를 30단계로 옮긴 gate_40을 실험군으로 설정

3. 핵심 지표, 보조 지표, 가드레일 지표를 각각 하나씩 정하고 이유를 설명하세요.
 - 핵심 지표는 단순 1일 평가보다는 7일 이후 시점을 나타내는 retention_7(재방문)
 - 보조 지표는 초기 반응 확인을 위해 retention_1로
 - 가드레일 지표는 활동이 감소하는지 확인하기 위해 sum_gamerounds(평균 게임 라운드 수)로 보겠다.

4. 요약표에서 차이가 보인다는 사실만으로 `gate_40`의 효과라고 결론 내릴 수 있나요?
 - ㄴㄴ 표본 변동으로 생길 수 있는 차이인지? 통계적으로 검정이 필요하다.

In [5]:
# 여기에 코드를 작성하세요.
duplicate_users = df["userid"].duplicated().sum()
versions = df["version"].unique()
group_counts = df["version"].value_counts().reindex(["gate_30", "gate_40"])

summary = (
    df.groupby('version').agg(
        users=("userid", "size"),
        avg_gamerounds=("sum_gamerounds", "mean"),
        retention_1=("retention_1", "mean"),
        retention_7=("retention_7", "mean")
    ).reindex(['gate_30', 'gate_40'])
)


print("userid 중복 수", duplicate_users)
print("version 고유값", versions)
print("\n[그룹별 사용자 수]")
print(group_counts)
print("\n[버전별 요약표]")
print(summary.round(4))

userid 중복 수 0
version 고유값 <StringArray>
['gate_30', 'gate_40']
Length: 2, dtype: str

[그룹별 사용자 수]
version
gate_30    44700
gate_40    45489
Name: count, dtype: int64

[버전별 요약표]
         users  avg_gamerounds  retention_1  retention_7
version                                                 
gate_30  44700         52.4563       0.4482       0.1902
gate_40  45489         51.2988       0.4423       0.1820


---

## 필수 2. 무작위 배정 실습과 그룹 비율 점검

실제 `version` 값은 변경하지 않고, 동일한 사용자 목록에 연습용 A/B 그룹을 새로 무작위 배정한 뒤 실제 배정 비율과 비교하세요.

### 수행 요구사항

1. `np.random.default_rng(42)`를 사용하세요.
2. 각 `userid`에 `A` 또는 `B`를 50:50 확률로 배정한 `randomized_users`를 만드세요.
3. 연습용 그룹별 인원수와 비율을 출력하세요.
4. 실제 `version`별 인원수와 비율도 출력하세요.
5. 실제 실험이 50:50 배정을 계획했다고 가정하고, 기대빈도를 전체 인원의 절반으로 설정하여 카이제곱 적합도 검정을 수행하세요.
6. 아래 질문에 답하세요.

### 질문

1. 시드를 고정하는 이유는 무엇인가요?
 - 무작위 과정을 재현하여 수강생과 검토자가 같은 배정 결과를 얻고 코드를 호가인할 수 있게 하기 위해서

2. 연습용 배정에서 한 사용자가 두 그룹에 동시에 포함되지 않았는지 어떻게 확인할 수 있나요?
 - userid별로 practice_group의 교유값 수를 계산하여 최대값이 1인지 확인한다.

3. 실제 배정의 카이제곱 검정 결과는 50:50 계획과 일치한다고 볼 수 있나요?
 - p-value가 0.05보다 작으므로 50:50 계획과 통계적으로 일치한다고 보기 어렵다.

4. `retention_1`, `retention_7`, `sum_gamerounds`를 랜덤화 이전 공변량의 균형 점검에 사용하면 안 되는 이유는 무엇인가요?
 - 세 변수는 버전을 경험한 이후의 측정된 결과이므로 처치영향을 받을 수 있다.
 - 균형점검에서는 실험 전에 측정된 기기, 국가 등 외부 요인 같은 공변량이 필요하지만 현 데이터에서는 제공하지 않았다.

In [6]:
# 여기에 코드를 작성하세요.
rng = np.random.default_rng(42)

randomized_users = df[["userid"]].copy()
randomized_users["practice_group"] = rng.choice(
    ["A", "B"], size=len(randomized_users), p=[0.5, 0.5]
)

practice_counts = randomized_users["practice_group"].value_counts().reindex(["A", "B"])
practice_ratios = randomized_users["practice_group"].value_counts(normalize=True).reindex(["A", "B"])

actual_counts = df["version"].value_counts().reindex(["gate_30", "gate_40"])
actual_ratios = df["version"].value_counts(normalize=True).reindex(["gate_30", "gate_40"])

expected_counts = np.repeat(len(df) / 2, 2)
chi2_srm, p_srm = stats.chisquare(
    actual_counts.to_numpy(), f_exp=expected_counts
)

max_groups_per_user = randomized_users.groupby("userid")["practice_group"].nunique().max()

print("[연습용 무작위 배정 인원수]")
print(practice_counts)

print("\n[연습용 무작위 배정 비율]")
print((practice_ratios * 100).round(3).astype(str) + "%")

print("\n[실제 version별 인원수]")
print(actual_counts)

print("\n[실제 version별 비율]")
print((actual_ratios * 100).round(3).astype(str) + "%")

print("\n[카이제곱 적합도 검정]")
print(f"카이제곱 통계량: {chi2_srm:.4f}")
print(f"p-value: {p_srm:.4f}")

print("\n사용자 1명당 최대 연습용 그룹 수:", max_groups_per_user)
print(f"50:50 적합도 검정: chi2 = {chi2_srm:.4f}, p-value = {p_srm:.4f}")

[연습용 무작위 배정 인원수]
practice_group
A    44852
B    45337
Name: count, dtype: int64

[연습용 무작위 배정 비율]
practice_group
A    49.731%
B    50.269%
Name: proportion, dtype: str

[실제 version별 인원수]
version
gate_30    44700
gate_40    45489
Name: count, dtype: int64

[실제 version별 비율]
version
gate_30    49.563%
gate_40    50.437%
Name: proportion, dtype: str

[카이제곱 적합도 검정]
카이제곱 통계량: 6.9024
p-value: 0.0086

사용자 1명당 최대 연습용 그룹 수: 1
50:50 적합도 검정: chi2 = 6.9024, p-value = 0.0086


---

## 과제. Cookie Cats A/B 테스트 전체 계획서 작성

`gate_30`을 기존 버전, `gate_40`을 새 버전으로 설정한 A/B 테스트 계획을 작성하세요.

### 수행 요구사항

1. 아래 네 단계를 모두 포함한 계획서를 작성하세요.
   - 가설 설정
   - 실험 설계
   - 실험 실행
   - 결과 분석
2. 실험 단위, 대조군, 실험군, 핵심·보조·가드레일 지표를 명시하세요.
3. 실행 전에 확인할 데이터 품질 항목을 두 가지 이상 작성하세요.
4. SUTVA 위반 또는 실험 간 간섭 가능성을 검토하세요.
5. 버전별 관측 지표를 다시 계산하되, 아직 통계적 검정을 하지 않았다는 점을 반영해 최종 의사결정을 보류하는 6~8문장의 결론을 작성하세요.

> 과제는 필수 문제와 동일한 수준입니다. 표본 크기나 MDE를 계산할 필요는 없습니다.


In [9]:
# 여기에 코드를 작성하세요.
duplicate_users = df["userid"].duplicated().sum()
missing_counts = df.isna().sum()
unique_versions = df["version"].unique()
negative_gamerounds = (df["sum_gamerounds"] < 0).any()

group_counts = df["version"].value_counts().reindex(["gate_30", "gate_40"])
group_ratios = df["version"].value_counts(normalize=True).reindex(["gate_30", "gate_40"])

summary = df.groupby("version").agg(
    users=("userid", "size"),
    avg_gamerounds=("sum_gamerounds", "mean"),
    retention_1=("retention_1", "mean"),
    retention_7=("retention_7", "mean")
)

print("userid 중복 수:", duplicate_users)
print("결측치 수:")
print(missing_counts)

print("\nversion 고유값:", unique_versions)
print("sum_gamerounds 음수 존재 여부:", bool(negative_gamerounds))

print("\n그룹별 인원수:")
print(group_counts)

print("\n그룹별 비율:")
print((group_ratios * 100).round(3))

print("\n버전별 요약표:")
print(summary.round(4))

userid 중복 수: 0
결측치 수:
userid            0
version           0
sum_gamerounds    0
retention_1       0
retention_7       0
dtype: int64

version 고유값: <StringArray>
['gate_30', 'gate_40']
Length: 2, dtype: str
sum_gamerounds 음수 존재 여부: False

그룹별 인원수:
version
gate_30    44700
gate_40    45489
Name: count, dtype: int64

그룹별 비율:
version
gate_30    49.563
gate_40    50.437
Name: proportion, dtype: float64

버전별 요약표:
         users  avg_gamerounds  retention_1  retention_7
version                                                 
gate_30  44700         52.4563       0.4482       0.1902
gate_40  45489         51.2988       0.4423       0.1820


### 1. 가설 설정
- 배경: 게임의 첫 번째 강제 대기 게이트를 30단계(gate_30)에서 40단계(gate_40)로 옮겼을 때 사용자 7일 재방문율(retention_7)이 달라지는지 확인한다.
- 귀무가설(H0): gate_30과 gate_40 사이에 7일 재방문율의 차이가 없다.
- 대립가설(H1): gate_30과 gate_40 사이에 7일 재방문율의 차이가 있다.

### 2. 실험 설계
- 실험 단위: userid(개별 사용자) > 한 사용자는 하나의 버전에만 배정되어 일관된 게임 경험을 제공받는다.
- 대조군: gate_30(기존 버전)
- 실험군: gate_40(새 버전)
- 핵심 지표: retention_7 > 장기적인 사용자 유지 여부 확인
- 보조 지표: retention_1 > 변경 직후의 초기 반응 확인
- 가드레일 지표: sum_gamerounds(평균 게임 라운드 수) > 게이트 위치 변경으로 사용자의 게임 활동량이 감소하지 않는지 확인
- 유의수준: alpha = 0.05

### 3. 실험 실행
#### 데이터 품질 점검
- userid 중복 여부 확인
- userid, version, sum_gamerounds, retention_1, retention_7 결측치 확인
- version 값이 gate_30와 gate_40으로만 구성되어 있는지 확인
- 그룹별 사용자 수와 배정 비율 확인(50:50)
- sum_gamerounds에 음수값 존재 여부 확인
#### SUTVA 및 실험 간 간섭 가능성 검토
- SUTVA가 성립하려면 한 사용자의 결과가 다른 사용자의 배정 상태에 영향받지 않아야 한다.
- 동일한 사용자가 재설치나 여러 기기를 통해 두 버전을 모두 경험한다면 처치가 섞여 SUTVA를 위반할 수 있으므로 userid 기준의 고정 배정이 필요하다.
- 또한 같은 기간에 다른 게임 업데이트, 이벤트, 프로모션 또는 별도의 A/B 테스트가 동시에 적용되면 이번 게이트 변경의 효과와 다른 실험의 효과가 섞일 수 있으므로 이러한 간섭도 가능한 한 통제해야 한다.

### 4. 결과 분석
- 1일 재방문율은 gate_30이 약 44.82%, gate_40이 약 44.23%이며 7일 재방문율은 각각 약 19.02%, 18.20%이다.
- 평균 게임 라운드 수도 gate_30이 약 52.46회, gate_40이 약 51.30회로 확인되었다.
- 다만 이 차이가 표본 변동에 의한 것인지 실제 효과인지는 아직 통계적으로 검정하지 않았으므로 관측값의 방향과 크기만으로 gate_40의 효과라고 판단해서는 안 된다.
- 게다가 필수2에서 확인한 카이제곱 적합도 검정 결과(p=0.0086)는 실제 배정 비율이 계획한 50:50과 다르다는 신호이므로 지표 차이를 해석하기 전에 실험 배정이나 데이터 수집 과정에 문제가 없었는지 먼저 점검해야 한다.

### 5. 결론
- 현재 관측치 기준으로 gate_40은 1일, 7일 재방문율과 평균 게임 라운드 수 세 지표 모두 gate_30보다 낮게 나타났다.
- 이러한 차이는 표본의 우연한 변동으로 발생했을 가능성이 있으므로 통계적 검정 없이는 gate_40의 효과를 판단할 수 없다.
- 게다가 실제 배정 비율에 대한 카이제곱 적합도 검정의 p-value가 0.05보다 작아 계획했던 50:50 배정과 실제 배정 사이에 표본 비율 불일치가 의심된다. 이에 추가적으로 실험 배정이나 데이터 수집 과정에서 문제가 없었는지 확인할 필요가 있다.
- 또한 사용자별 버전이 일관되게 유지되었는지와 다른 실험이나 게임 업데이트 등 외부 요인에 의한 간섭이 있었는지도 점검해야 한다.
- 이후 1일, 7일 재방문율과 평균 게임 라운드 수에 대해 적절한 통계적 검정을 실시하고 차이의 크기와 실질적 의미도 함께 확인해야 한다.
- 현재 단계에서는 gate_40의 최종 적용 여부를 보류하고 추가 검정과 실험 품질 점검 결과를 바탕으로 결정하는 것이 적절하다.

---

## 실습 마무리

- 어떤 문제가 있었는가?
> 그룹별 유지율과 평균 게임 라운드만 비교하여 사용자의 고정 배정 여부, 계획한 표본 비융ㄹ, 지표 사전 지정,
  외부 간섭, 로ㅓ깅 등의 문제를 놓칠 수 있었다.

- 어떻게 개선했는가?
> 실험 단위를 userid로 명시하고 대조군, 실험군과 3가지 지표를 사전에 정의했다.
> 실제 배정 비율, 이용자 중복 체크, 50:50 적합도, 결과변수, 사전 공변량의 차이 등을 함께 점검했다.

- 무엇을 근거로 개선되었다고 판단했는가?
> 사용자 중복 0건, 실제 그룹 비율 약 49.5%, 50.4%, 적합도 검정 p-value가 0.05 미만을 확인하여
  단순히 비슷해 보인다는 필수1의 판단보다는 구체적인 조사 증거를 확보했다.
> 40gate를 배포 전에 기술 검정을 통해 버전 업 배포 결정을 보류했다.

단순한 그룹별 지표 비교에서 놓칠 수 있는 문제와, 실험 단위·사전 지표·배정 비율·간섭 가능성을 명시하면서 설계가 어떻게 개선되었는지 정리하세요.
